# 38. Language models, pretraining, and next-token prediction

**Level:** Advanced

**Prerequisites:** 27-28 and 35-37: optimization, attention, and tokens.

[Course index](00_START_HERE.ipynb) | [Previous](37_Tokens_Subwords_and_Context_Windows.ipynb) | [Next](39_Inference_Temperature_TopP_and_Generation.ipynb)

**Topics:** LLM, Pretraining, Next-token prediction, Language modeling, Cross-entropy, Perplexity.

**Matching script:** `examples\38_Language_Model_Pretraining_and_Next_Tokens.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A causal language model estimates the next-token distribution conditioned on earlier tokens.
During pretraining, a sequence provides many shifted input/target pairs. The target is derived
from the text itself, so next-token prediction is self-supervised even though it uses a label-like
objective.

Modern LLMs learn contextual representations through many transformer layers. A bigram model
conditions only on the preceding token; it cannot represent the long contexts or capabilities
of a large transformer. Starting with a trained bigram makes the objective easy to inspect.

Cross-entropy measures negative log probability of the observed target. Perplexity is exp(mean
negative log likelihood) under matching tokenization and evaluation conditions. Comparing values
across tokenizers or datasets can be misleading.

Pretraining does not itself establish instruction following, factual correctness, permission
to reuse source data, or safe tool behavior. Post-training may add instruction and preference
objectives. Train/test contamination can inflate evaluations when benchmarks appear in training.

## Mathematics or data contract

For tokens t_1..t_n, a causal objective sums -log P(t_(i+1) | t_1..t_i).

## Important limitation

The local bigram is a real trained neural language model, but is not an LLM or a useful assistant.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Train a tiny neural bigram from scratch

**Purpose:** Shift targets and optimize next-token loss without downloading weights.

### Read the code in this order
1. Encode a deliberately simple repeated token stream.
2. Construct one-token input and next-token targets.
3. Train an embedding table of next-token logits.
4. Verify loss decreases and learned next-token transitions fit the training pattern.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
training loss: 0.831 -> 0.006
learned transitions: {'a': 'b', 'b': 'c', 'c': 'a'}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
text = "abc" * 60
vocabulary = sorted(set(text))
token_to_id = {token: index for index, token in enumerate(vocabulary)}
ids = torch.tensor([token_to_id[token] for token in text])
inputs, targets = ids[:-1], ids[1:]
model = nn.Embedding(len(vocabulary), len(vocabulary))
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.1), nn.CrossEntropyLoss()
initial = criterion(model(inputs), targets).item()
for _ in range(90):
    optimizer.zero_grad()
    loss = criterion(model(inputs), targets)
    loss.backward()
    optimizer.step()
assert loss.item() < initial and loss.item() < 0.05
next_ids = model(torch.arange(3)).argmax(dim=1).tolist()
assert next_ids == [token_to_id["b"], token_to_id["c"], token_to_id["a"]]
print("training loss:", round(initial, 3), "->", round(loss.item(), 3))
print("learned transitions:", {token: vocabulary[next_ids[index]] for index, token in enumerate(vocabulary)})

training loss: 0.831 -> 0.006
learned transitions: {'a': 'b', 'b': 'c', 'c': 'a'}


### Interpret example 1

The result should demonstrate this purpose: **Shift targets and optimize next-token loss without downloading weights.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Train a miniature causal transformer

**Purpose:** Combine embeddings, positions, a causal attention block, and next-token cross-entropy in a real tiny training run.

### Read the code in this order
1. Encode a repeated synthetic pattern with shifted targets.
2. Create learned token/position embeddings and a masked block.
3. Train the vocabulary head end-to-end.
4. Verify decreasing training loss without claiming held-out language ability.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
tiny transformer's training loss: 1.501 -> 0.0
trained on a repeating synthetic pattern, not natural language
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)

class TinyCausalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens, self.positions = nn.Embedding(4, 8), nn.Embedding(6, 8)
        self.block = nn.TransformerEncoderLayer(8, 2, 16, dropout=0.0, batch_first=True)
        self.head = nn.Linear(8, 4)

    def forward(self, ids):
        length = ids.shape[1]
        hidden = self.tokens(ids) + self.positions(torch.arange(length))
        mask = torch.triu(torch.ones(length, length, dtype=torch.bool), diagonal=1)
        return self.head(self.block(hidden, src_mask=mask))

ids = torch.tensor([[0, 1, 2, 3, 0, 1], [1, 2, 3, 0, 1, 2]])
targets = (ids + 1) % 4
model = TinyCausalModel()
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.03), nn.CrossEntropyLoss()
initial = criterion(model(ids).reshape(-1, 4), targets.reshape(-1)).item()
for _ in range(90):
    optimizer.zero_grad()
    loss = criterion(model(ids).reshape(-1, 4), targets.reshape(-1))
    loss.backward()
    optimizer.step()
assert loss.item() < initial and loss.item() < 0.1
print("tiny transformer's training loss:", round(initial, 3), "->", round(loss.item(), 3))
print("trained on a repeating synthetic pattern, not natural language")

tiny transformer's training loss: 1.501 -> 0.0
trained on a repeating synthetic pattern, not natural language


### Interpret example 2

The result should demonstrate this purpose: **Combine embeddings, positions, a causal attention block, and next-token cross-entropy in a real tiny training run.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Split a larger local text into disjoint documents and report held-out loss instead of memorization loss.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `LLM` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For tokens t_1..t_n, a causal objective sums -log P(t_(i+1) | t_1..t_i).
- Name the boundary of the demonstration: The local bigram is a real trained neural language model, but is not an LLM or a useful assistant.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/1706.03762`
- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.